In [1]:
import pandas as pd

movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

print(movies.shape)
print(ratings.shape)

movies.head()

(9742, 3)
(100836, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [2]:
movie_ratings = ratings.merge(
    movies,
    on="movieId"
)

movie_ratings.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [3]:
print(movie_ratings.shape)

movie_ratings.sample(5)

(100836, 6)


,userId,movieId,rating,timestamp,title,genres
30719,215,68358,4.0,1260908615,Star Trek (2009),Action|Adventure|Sci-Fi|IMAX
69162,448,3198,4.0,1019137809,Papillon (1973),Crime|Drama
37842,256,60069,4.5,1446579979,WALL·E (2008),Adventure|Animation|Children|Romance|Sci-Fi
35792,242,185,3.0,834073137,"Net, The (1995)",Action|Crime|Thriller
60487,391,1029,3.0,1030824424,Dumbo (1941),Animation|Children|Drama|Musical


In [4]:
movie_stats = movie_ratings.groupby("title").agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()

movie_stats.head()

,title,avg_rating,rating_count
0,'71 (2014),4.0,1
1,'Hellboy': The Seeds of Creation (2004),4.0,1
2,'Round Midnight (1986),3.5,2
3,'Salem's Lot (2004),5.0,1
4,'Til There Was You (1997),4.0,2


In [5]:
popular_movies = movie_stats[
    movie_stats["rating_count"] >= 50
].sort_values(
    by="avg_rating",
    ascending=False
)

popular_movies.head(20)

,title,avg_rating,rating_count
7593,"Shawshank Redemption, The (1994)",4.429022,317
3499,"Godfather, The (1972)",4.289062,192
3011,Fight Club (1999),4.272936,218
1961,Cool Hand Luke (1967),4.271930,57
2531,Dr. Strangelove or: How I Learned to Stop Worr...,4.268041,97
6999,Rear Window (1954),4.261905,84
3500,"Godfather: Part II, The (1974)",4.259690,129
2334,"Departed, The (2006)",4.252336,107
3564,Goodfellas (1990),4.250000,126
1593,Casablanca (1942),4.240000,100


In [6]:
user_movie_matrix = movie_ratings.pivot_table(
    index="userId",
    columns="title",
    values="rating"
)

user_movie_matrix.shape

(610, 9719)

## Business Insight

The Shawshank Redemption received the highest average rating among movies with at least 50 ratings, indicating strong user satisfaction and broad audience appeal.

The user-movie matrix transformed 100,836 ratings into a structure suitable for recommendation modeling, with 610 users and 9,719 movies.

In [7]:
# Collaborative Filtering Recommendations

In [8]:
movie_ratings = user_movie_matrix["Toy Story (1995)"]

movie_ratings.head()

userId
1    4.0
2    NaN
3    NaN
4    NaN
5    4.0
Name: Toy Story (1995), dtype: float64

In [9]:
similar_movies = user_movie_matrix.corrwith(movie_ratings)

similar_movies.head()

C:\Users\jmerc\netflix-recommendation-engine\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
C:\Users\jmerc\netflix-recommendation-engine\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
C:\Users\jmerc\netflix-recommendation-engine\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
C:\Users\jmerc\netflix-recommendation-engine\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\jmerc\netflix-recommendation-engine\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


title
'71 (2014)                                NaN
'Hellboy': The Seeds of Creation (2004)   NaN
'Round Midnight (1986)                    NaN
'Salem's Lot (2004)                       NaN
'Til There Was You (1997)                 NaN
dtype: float64

In [10]:
similar_movies = user_movie_matrix.corrwith(movie_ratings)

similar_movies.head()

title
'71 (2014)                                NaN
'Hellboy': The Seeds of Creation (2004)   NaN
'Round Midnight (1986)                    NaN
'Salem's Lot (2004)                       NaN
'Til There Was You (1997)                 NaN
dtype: float64

In [11]:
corr_toy_story = pd.DataFrame(similar_movies, columns=["Correlation"])

corr_toy_story.dropna(inplace=True)

corr_toy_story.head()

,Correlation
title,
"'burbs, The (1989)",0.240563
(500) Days of Summer (2009),0.353833
*batteries not included (1987),-0.427425
10 Cent Pistol (2015),1.000000
10 Cloverfield Lane (2016),-0.285732


In [12]:
corr_toy_story = corr_toy_story.join(movie_stats["rating_count"])

corr_toy_story.head()

,Correlation,rating_count
title,,
"'burbs, The (1989)",0.240563,NaN
(500) Days of Summer (2009),0.353833,NaN
*batteries not included (1987),-0.427425,NaN
10 Cent Pistol (2015),1.000000,NaN
10 Cloverfield Lane (2016),-0.285732,NaN


In [13]:
recommendations = corr_toy_story[
    corr_toy_story["rating_count"] > 100
].sort_values(
    "Correlation",
    ascending=False
)

recommendations.head(10)

,Correlation,rating_count
title,,


In [14]:
movie_stats_fixed = movie_ratings.groupby("title").agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count")
)

corr_toy_story = pd.DataFrame(similar_movies, columns=["Correlation"])

corr_toy_story.dropna(inplace=True)

corr_toy_story = corr_toy_story.join(movie_stats_fixed["rating_count"])

recommendations = corr_toy_story[
    corr_toy_story["rating_count"] > 100
].sort_values(
    "Correlation",
    ascending=False
)

recommendations.head(10)

KeyError: 'title'

In [15]:
movie_ratings.head()

userId
1    4.0
2    NaN
3    NaN
4    NaN
5    4.0
Name: Toy Story (1995), dtype: float64

In [16]:
type(movie_ratings)

pandas.Series

In [17]:
globals().keys()

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__session__', 'platform', 'sys', 'original_ps1', 'is_wsl', 'REPLHooks', 'get_last_command', 'PS1', '_i', '_ii', '_iii', '_i1', 'pd', 'movies', 'ratings', '_1', '_i2', 'movie_ratings', '_2', '_i3', '_3', '_i4', 'movie_stats', '_4', '_i5', 'popular_movies', '_5', '_i6', 'user_movie_matrix', '_6', '_i7', '_i8', '_8', '_i9', 'similar_movies', '_9', '_i10', '_10', '_i11', 'corr_toy_story', '_11', '_i12', '_12', '_i13', 'recommendations', '_13', '_i14', '_i15', '_15', '_i16', '_16', '_i17'])

In [18]:
ratings_with_movies = ratings.merge(movies, on="movieId")

movie_stats_fixed = ratings_with_movies.groupby("title").agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count")
)

toy_story_ratings = user_movie_matrix["Toy Story (1995)"]

similar_movies = user_movie_matrix.corrwith(toy_story_ratings)

corr_toy_story = pd.DataFrame(similar_movies, columns=["Correlation"])
corr_toy_story.dropna(inplace=True)

corr_toy_story = corr_toy_story.join(movie_stats_fixed["rating_count"])

recommendations = corr_toy_story[
    corr_toy_story["rating_count"] > 100
].sort_values(
    "Correlation",
    ascending=False
)

recommendations.head(10)

C:\Users\jmerc\netflix-recommendation-engine\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
C:\Users\jmerc\netflix-recommendation-engine\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
C:\Users\jmerc\netflix-recommendation-engine\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
C:\Users\jmerc\netflix-recommendation-engine\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\jmerc\netflix-recommendation-engine\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,Correlation,rating_count
title,,
Toy Story (1995),1.000000,215
"Incredibles, The (2004)",0.643301,125
Finding Nemo (2003),0.618701,141
Aladdin (1992),0.611892,183
"Monsters, Inc. (2001)",0.490231,132
Mrs. Doubtfire (1993),0.446261,144
"Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)",0.438237,120
American Pie (1999),0.420117,103
Die Hard: With a Vengeance (1995),0.410939,144
